# Day 07: vLLM — PagedAttention & Continuous Batching
> *Inference Engineering* — Chapter 4.3.1 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 05 (Kernel Fusion)


## Concept Overview

vLLM introduced two transformative ideas for LLM serving: PagedAttention and continuous batching. PagedAttention manages KV cache memory in fixed-size blocks (like OS virtual memory pages), eliminating fragmentation and enabling memory sharing across requests with common prefixes. Continuous batching (iteration-level scheduling) adds new requests to the batch as soon as slots free up — rather than waiting for all requests in a batch to finish — dramatically improving GPU utilization. Together, these enable throughput improvements of 2-24x over naive serving.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. KV Cache Fragmentation Problem

In naive KV cache management, each request gets a contiguous memory region pre-allocated for its max sequence length. This leads to internal fragmentation (reserved but unused space) and prevents memory sharing. PagedAttention maps logical KV 'pages' to physical blocks, like virtual memory.


In [ ]:
import matplotlib.patches as mpatches

def visualize_fragmentation():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Naive allocation: requests of different lengths in contiguous blocks
    total_mem = 100
    requests = [('Req A', 60, 20), ('Req B', 30, 28), ('Req C', 40, 5)]
    # (name, reserved, used)

    y = 0
    colors = ['#4C72B0','#DD8452','#55A868']
    for (name, reserved, used), color in zip(requests, colors):
        ax1.barh(y, reserved, height=0.6, color=color, alpha=0.3, label=f'{name} (reserved)')
        ax1.barh(y, used, height=0.6, color=color, alpha=0.9)
        ax1.text(used+1, y, f'{used}/{reserved} used', va='center', fontsize=9)
        y += 1

    ax1.set_xlim(0, total_mem)
    ax1.set_yticks(range(len(requests)))
    ax1.set_yticklabels([r[0] for r in requests])
    ax1.set_xlabel('KV Cache Memory (tokens)')
    ax1.set_title('Naive KV Cache: Fragmentation')
    waste = sum(r-u for _,r,u in requests)
    ax1.text(50, -0.7, f'Total waste: {waste} tokens ({waste/total_mem*100:.0f}%)', ha='center', color='red', fontsize=10)

    # PagedAttention: blocks allocated on demand
    block_size = 16
    num_blocks = 8
    block_assignments = {'Req A': [0,1,2], 'Req B': [3,4], 'Req C': [5]}
    block_colors_map = {'Req A':'#4C72B0','Req B':'#DD8452','Req C':'#55A868', 'Free':'lightgray'}

    for i in range(num_blocks):
        owner = 'Free'
        for req, blocks in block_assignments.items():
            if i in blocks:
                owner = req
        color = block_colors_map[owner]
        ax2.barh(0, 1, left=i, height=0.5, color=color, edgecolor='black')
        ax2.text(i+0.5, 0, f'B{i}', ha='center', va='center', fontsize=8)

    ax2.set_xlim(0, num_blocks)
    ax2.set_ylim(-0.5, 0.5)
    ax2.set_xlabel('Physical KV Blocks')
    ax2.set_title('PagedAttention: Block Allocation')
    legend_patches = [mpatches.Patch(color=v, label=k) for k,v in block_colors_map.items()]
    ax2.legend(handles=legend_patches, loc='upper right')
    ax2.set_yticks([])

    plt.tight_layout()
    plt.savefig('paged_attention.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved paged_attention.png')

visualize_fragmentation()


## 2. Continuous Batching Simulation

Static batching waits for all requests in a batch to complete before starting new ones. Continuous batching (iteration-level scheduling) inserts new requests at each decode step when slots free up. This eliminates the 'head-of-line blocking' where short requests wait for a long one to finish.


In [ ]:
import heapq

def simulate_batching(strategy, requests, max_batch=4, seed=42):
    """Simulate static vs continuous batching. Returns GPU idle time."""
    random.seed(seed)
    pending = list(requests)  # (arrival_time, duration)
    time = 0
    active = []
    completions = []
    gpu_busy = 0
    events = []

    while pending or active:
        # Add available requests
        if strategy == 'continuous':
            while pending and pending[0][0] <= time and len(active) < max_batch:
                arr, dur = pending.pop(0)
                active.append({'start': time, 'remaining': dur, 'original_dur': dur})
        elif strategy == 'static':
            if not active and pending:
                batch = []
                while pending and len(batch) < max_batch:
                    arr, dur = pending.pop(0)
                    batch.append({'start': time, 'remaining': dur, 'original_dur': dur})
                active = batch

        if not active:
            time += 1
            continue

        # Run one decode step
        for req in active:
            req['remaining'] -= 1
        gpu_busy += 1
        time += 1

        # Remove completed
        done = [r for r in active if r['remaining'] <= 0]
        active = [r for r in active if r['remaining'] > 0]
        for r in done:
            completions.append(time - r['start'])

    return time, gpu_busy, completions

# Generate requests: arrivals at t=0..10, durations 5-20 steps
np.random.seed(42)
arrival_times = sorted(np.random.randint(0, 8, 12).tolist())
durations = np.random.randint(5, 20, 12).tolist()
requests = list(zip(arrival_times, durations))

for strategy in ['static', 'continuous']:
    total_time, busy, latencies = simulate_batching(strategy, requests)
    utilization = busy / total_time * 100
    print(f'{strategy.capitalize()} batching:')
    print(f'  Total time: {total_time} steps')
    print(f'  GPU utilization: {utilization:.0f}%')
    print(f'  Mean latency: {np.mean(latencies):.1f} steps')
    print(f'  P99 latency: {np.percentile(latencies,99):.1f} steps')
    print()


## 3. Block Table and Memory Management

PagedAttention uses a block table mapping logical sequence positions to physical GPU memory blocks. This enables copy-on-write prefix sharing: multiple requests can reference the same physical blocks for a shared system prompt.


In [ ]:
class BlockAllocator:
    def __init__(self, num_blocks, block_size):
        self.block_size = block_size
        self.free_blocks = list(range(num_blocks))
        self.allocated = {}  # req_id -> [block_ids]
        self.num_blocks = num_blocks

    def allocate(self, req_id, num_tokens):
        needed = (num_tokens + self.block_size - 1) // self.block_size
        if needed > len(self.free_blocks):
            return None  # OOM
        blocks = [self.free_blocks.pop() for _ in range(needed)]
        self.allocated[req_id] = blocks
        return blocks

    def free(self, req_id):
        if req_id in self.allocated:
            self.free_blocks.extend(self.allocated.pop(req_id))

    def utilization(self):
        used = self.num_blocks - len(self.free_blocks)
        return used / self.num_blocks

# Simulate a serving session
allocator = BlockAllocator(num_blocks=32, block_size=16)
print('Block allocator simulation:')
print(f'Total blocks: 32, block size: 16 tokens (512 tokens total capacity)')
print()

scenario = [
    ('alloc', 'req1', 48),
    ('alloc', 'req2', 96),
    ('alloc', 'req3', 32),
    ('free',  'req1', None),
    ('alloc', 'req4', 64),
]
for op, req, tokens in scenario:
    if op == 'alloc':
        blocks = allocator.allocate(req, tokens)
        print(f'  Alloc {req} ({tokens} tokens) → blocks {blocks}, util={allocator.utilization():.0%}')
    else:
        allocator.free(req)
        print(f'  Free  {req} → util={allocator.utilization():.0%}')


## Experiments: Try These

1. **vLLM installation**: Deploy vLLM on spark-01 (`ssh nvidia@192.168.1.76`) and compare TTFT + throughput vs a naive loop.
2. **Block size sensitivity**: Modify the BlockAllocator to use block sizes 8, 16, 32, 64. Plot fragmentation vs block size.
3. **Prefix sharing**: Extend the BlockAllocator with a prefix cache (hash → block_id). Simulate 100 requests with a shared system prompt and measure memory savings.


## Key Takeaways

- PagedAttention eliminates KV cache fragmentation by using fixed-size blocks mapped via a block table, enabling >99% memory utilization.
- Continuous batching replaces per-batch scheduling with per-iteration scheduling, filling GPU slots as requests complete rather than waiting for full batch turnover.
- Copy-on-write block sharing allows multiple requests with common prefixes (e.g., system prompt) to share physical KV cache blocks.
- Together, these two ideas give vLLM 2-24x higher throughput than naive serving at the same latency budget.

## References
- *Inference Engineering* Ch 4.3.1 — Philip Kiely, Baseten Books 2026
- Kwon et al. (2023), "Efficient Memory Management for Large Language Model Serving with PagedAttention"
